# Phase 4 — RT-DETR-X

Phase 3 confirmed the ensemble route wasn't going to work with the models I had. Going back to the single-model line of attack: just scale up.

RT-DETR-L (Exp C, 0.4484 mAP) is the current best by a clear margin. The X variant is the same architecture with a wider encoder/decoder — ~65M params vs ~32M. Same training pipeline, same data, same augmentation. It's the lowest-risk, highest-expected-return thing on the table.

| Model | Val mAP@0.50:0.95 |
|-------|-------------------|
| YOLO baseline (Phase 1) | 0.4285 |
| YOLO + 4ch (Phase 2 Exp B) | 0.4293 |
| RT-DETR-L (Phase 2 Exp C) | **0.4484** |

**Target for this phase:** beat 0.4484. Stretch goal: 0.50.

## Training strategy

The big question is what to change from the L recipe. A few things X needs that L didn't:
- **Bigger warmup** — larger models are more sensitive to the first few epochs, so 10 warmup epochs instead of 5
- **Lower batch size** — VRAM-bound on the 5080 at 16 GB. Batch 6 keeps headroom; batch 8 OOMs once mosaic + copy_paste fire on dense images
- **`amp=False` is mandatory** — found this the hard way during a first attempt: AMP produces NaN losses with RT-DETR-X at any learning rate, no matter the warmup. Float32 only

The trickier choice was the LR schedule. RT-DETR's documented recipe is AdamW with a specific cosine schedule, but Ultralytics' `optimizer="auto"` silently overrides `lr0` and picks its own peak — which I confirmed when an earlier attempt accidentally trained at an effective peak of ~2e-3 despite me setting `lr0=1e-5`. So this run sets `optimizer="AdamW"` explicitly and uses `lr0=1e-3` with `lrf=0.001` for the cosine tail down to ~1e-6 by the end of training.

## Things I'm not going to retry
- imgsz=800 (strictly worse than 640 in every prior run)
- The 4th positional channel (Phase 2 already established it adds noise)
- WBF with another YOLO (Phase 3 established consensus denominator is the problem)
- TTA on YOLO (Phase 2 already established it hurts)

In [3]:
import random, numpy as np, torch
'''
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
'''

'\nSEED = 42\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntorch.manual_seed(SEED)\ntorch.cuda.manual_seed_all(SEED)\ntorch.backends.cudnn.deterministic = True\ntorch.backends.cudnn.benchmark     = False\n'

## Setup

Same paths and split as previous phases.

In [4]:
import json, gc, tempfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch

from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ── Paths ─────────────────────────────────────────────────────────────────────
CLEARSAR  = Path(".")
ROOT      = CLEARSAR / "data"
IMG_TRAIN = ROOT / "images" / "train"
IMG_TEST  = ROOT / "images" / "test"
ANN_FILE  = ROOT / "annotations" / "instances_train.json"
YAML_PATH = CLEARSAR / "clearsar.yaml"
VAL_TXT   = CLEARSAR / "val_split.txt"

EXP_C_BEST = Path("runs/clearsar/exp_c_rtdetr_l/weights/best.pt")

assert EXP_C_BEST.exists(), f"Exp C checkpoint not found: {EXP_C_BEST}"
print("Exp C checkpoint (RT-DETR-L):", EXP_C_BEST)
print("Train images:", len(list(IMG_TRAIN.glob("*.png"))))
print("Test  images:", len(list(IMG_TEST.glob("*.png"))))

Exp C checkpoint (RT-DETR-L): runs\clearsar\exp_c_rtdetr_l\weights\best.pt
Train images: 3154
Test  images: 786


## Experiment D — RT-DETR-X from COCO

Starting from `rtdetr-x.pt` (COCO pretrained). Augmentation is identical to Exp C — that part already worked, no reason to change it.

The recipe in detail:

| Param | Value | Why |
|-------|-------|-----|
| `optimizer` | `AdamW` (explicit) | Never let `"auto"` pick — it silently overrides `lr0` |
| `lr0` | `1e-3` | Matches RT-DETR's documented recipe |
| `lrf` | `0.001` | Cosine tail collapses LR to ~1e-6 by the end |
| `warmup_epochs` | 10 | X needs more warmup than L did (5) |
| `epochs` | 150 | More capacity → more steps to converge |
| `patience` | 70 | Generous — better to let it run too long than stop too early |
| `batch` | 6 | VRAM-bound at ~13 GB peak |
| `amp` | `False` | **Mandatory** — AMP produces NaN losses with RT-DETR-X |

This is going to take roughly 12 hours on the 5080. Time to go for a walk.

In [6]:
model_d = YOLO("rtdetr-x.pt")

results_d = model_d.train(
    data            = str(YAML_PATH.resolve()),
    epochs          = 150,
    imgsz           = 640,
    amp             = False,        # mandatory — AMP causes NaN on RT-DETR-X
    batch           = 6,
    workers         = 4,
    device          = 0,
    project         = "runs/clearsar",
    name            = "exp_d_rtdetr_x",
    exist_ok        = True,
    seed            = False,
    deterministic   = True,
    # ── Augmentation (identical to original Exp D — this part worked) ─────
    degrees         = 0.0,          # no rotation — stripes are strictly horizontal
    flipud          = 0.0,          # no vertical flip — breaks stripe semantics
    fliplr          = 0.5,          # horizontal flip is safe
    mosaic          = 1.0,
    close_mosaic    = 10,
    copy_paste      = 0.1,          # helps sparse-label images
    mixup           = 0.0,
    hsv_h           = 0.0,          # SAR quicklooks have no meaningful hue
    hsv_s           = 0.3,
    hsv_v           = 0.4,
    # ── Optimizer & LR schedule ───────────────────────────────────────────
    optimizer       = "AdamW",      # EXPLICIT — don't let "auto" pick silently
    lr0             = 1e-3,         # matches Exp D's real effective peak regime
    lrf             = 0.001,         # cosine floor → 1e-5 at e100
    cos_lr          = True,
    warmup_epochs   = 10,           # same as original Exp D
    warmup_bias_lr  = 0.0,
    warmup_momentum = 0.937,
    patience        = 70,           # matches original Exp D budget
    save            = True,
    save_period     = 15,           # more intermediate checkpoints for diagnosis
    val             = True,
    plots           = True,
    verbose         = True,
)

print("\n=== Exp D (restructured) done ===")

New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.37  Python-3.13.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\Victor\Desktop\Projects\ESA_SAR\ClearSAR\clearsar.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.3, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-x.pt, momentu

c:\Users\Victor\.conda\envs\esa_sar\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/150      11.6G      1.114     0.8811     0.4965          4        640: 100% ━━━━━━━━━━━━ 474/474 3.0it/s 2:36<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 6.4it/s 4.2s0.2s
                   all        315        885      0.701      0.593      0.644      0.374

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/150        12G     0.5753     0.7166     0.2344         37        640: 0% ──────────── 0/474  0.4s

c:\Users\Victor\.conda\envs\esa_sar\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/150        12G     0.6072     0.6251     0.1786          7        640: 100% ━━━━━━━━━━━━ 474/474 3.2it/s 2:27<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 6.5it/s 4.2s0.2s
                   all        315        885      0.691      0.574      0.651      0.379

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/150      11.9G     0.7895     0.5721     0.2457         23        640: 0% ──────────── 0/474  0.4s

...

    127/150      11.9G     0.3727     0.4866    0.07533          1        640: 100% ━━━━━━━━━━━━ 474/474 3.6it/s 2:11<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 6.6it/s 4.1s0.2s
                   all        315        885      0.705      0.675      0.673      0.403

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    128/150      11.9G     0.2296     0.3994    0.06988         10        640: 0% ──────────── 0/474  0.3s

c:\Users\Victor\.conda\envs\esa_sar\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    128/150      11.9G     0.3637     0.4819    0.07586          1        640: 100% ━━━━━━━━━━━━ 474/474 3.6it/s 2:11<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 6.6it/s 4.1s0.2s
                   all        315        885      0.679       0.68      0.659      0.394

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    129/150      11.9G     0.5498     0.5462     0.1203         25        640: 0% ──────────── 0/474  0.3s

c:\Users\Victor\.conda\envs\esa_sar\Lib\site-packages\torch\autograd\graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:164.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    129/150      11.9G     0.3748     0.4814    0.07618          3        640: 100% ━━━━━━━━━━━━ 474/474 3.6it/s 2:11<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 6.6it/s 4.1s0.2s
                   all        315        885      0.675      0.675      0.655      0.388
EarlyStopping: Training stopped early as no improvement observed in last 70 epochs. Best results observed at epoch 59, best model saved as best.pt.
To update EarlyStopping(patience=70) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

129 epochs completed in 5.045 hours.
Optimizer stripped from C:\Users\Victor\Desktop\Projects\ESA_SAR\ClearSAR\runs\detect\runs\clearsar\exp_d_rtdetr_x\weights\last.pt, 135.5MB
Optimizer stripped from C:\Users\Victor\Desktop\Projects\ESA_SAR\ClearSAR\runs\detect\runs\clearsar\exp_d_rtdetr_x\weights\best.pt, 135.5MB

Validating C:\Users\Victor\Desktop\Projects\ESA_SAR\ClearSAR

## Validation — does X beat L?

Loading the best checkpoint and comparing against Exp C's RT-DETR-L number directly.

In [8]:
ckpt_d = Path("runs/clearsar/exp_d_rtdetr_x/weights/best.pt")
model_d_best = YOLO(str(ckpt_d))

metrics_d = model_d_best.val(
    data   = str(YAML_PATH.resolve()),
    imgsz  = 640,
    batch  = 4,
    device = 0,
    plots  = True,
)

map_d   = metrics_d.box.map
map50_d = metrics_d.box.map50
map75_d = metrics_d.box.map75

print(f"\n{'Model':<28} {'mAP@0.50:0.95':>14} {'mAP@0.50':>9} {'mAP@0.75':>9} {'Δ vs Exp C':>11}")
print("-" * 75)
print(f"{'Exp C  RT-DETR-L@640':<28} {0.4484:>14.4f} {0.7353:>9.4f} {0.4880:>9.4f} {'(baseline)':>11}")
print(f"{'Exp D  RT-DETR-X@640':<28} {map_d:>14.4f} {map50_d:>9.4f} {map75_d:>9.4f} {map_d - 0.4484:>+11.4f}")

if map_d > 0.4484:
    print(f"\n>>> RT-DETR-X is the new best model  (+{map_d - 0.4484:.4f} over RT-DETR-L)")
    print(f">>> Gap to target (0.50): {0.50 - map_d:.4f}")
else:
    print(f"\n>>> RT-DETR-X did not improve over RT-DETR-L ({map_d - 0.4484:+.4f})")
    print(f">>> Keep RT-DETR-L as best model. Consider Priority 2 (longer training).")

Ultralytics 8.4.37  Python-3.13.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
rt-detr-x summary: 378 layers, 65,469,491 parameters, 0 gradients, 222.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 3756.51092.9 MB/s, size: 247.7 KB)
val: Scanning C:\Users\Victor\Desktop\Projects\ESA_SAR\ClearSAR\data\labels\train.cache... 315 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 315/315 188.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 79/79 20.4it/s 3.9s0.1s
                   all        315        885      0.707      0.718      0.738       0.45
Speed: 0.5ms preprocess, 9.6ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to C:\Users\Victor\Desktop\Projects\ESA_SAR\ClearSAR\runs\detect\val12

Model                         mAP@0.50:0.95  mAP@0.50  mAP@0.75  Δ vs Exp C
---------------------------------------------------------------------------
Exp C  RT-DETR-L@640        

In [9]:
del model_d, model_d_best
gc.collect()
torch.cuda.empty_cache()

## Confidence threshold sweep

Quick check that the optimal confidence for X is in the same range as L (which peaked at 0.05–0.10). Going to sweep a bit wider this time, down to 0.01, because COCO mAP rewards keeping low-confidence candidates around.

In [10]:
with open(ANN_FILE) as f:
    coco_gt_raw = json.load(f)

val_stems = set(Path(p.strip()).stem for p in VAL_TXT.read_text().splitlines() if p.strip())
val_meta  = [img for img in coco_gt_raw["images"] if Path(img["file_name"]).stem in val_stems]
val_ids   = {img["id"] for img in val_meta}
val_anns  = [a for a in coco_gt_raw["annotations"] if a["image_id"] in val_ids]
val_paths = [Path(p.strip()) for p in VAL_TXT.read_text().splitlines() if p.strip()]

gt_tmp = Path(tempfile.gettempdir()) / "val_gt_phase4.json"
gt_tmp.write_text(json.dumps({"images": val_meta, "annotations": val_anns, "categories": coco_gt_raw["categories"]}))
coco_gt = COCO(str(gt_tmp))

print(f"Val images: {len(val_meta)}  |  Val annotations: {len(val_anns)}")

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Val images: 315  |  Val annotations: 885


In [16]:
model_sweep = YOLO(str(ckpt_d))

CONF_VALUES  = [0.01, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]
IOU_NMS      = 0.5
conf_results = []

for conf in CONF_VALUES:
    dets = []
    for i in range(0, len(val_paths), 16):
        preds = model_sweep.predict(
            source  = [str(p) for p in val_paths[i:i+16]],
            imgsz   = 640,
            conf    = conf,
            iou     = IOU_NMS,
            device  = 0,
            verbose = False,
        )
        for result, img_path in zip(preds, val_paths[i:i+16]):
            image_id = int(img_path.stem)
            if image_id not in val_ids:
                continue
            for (x1, y1, x2, y2), score in zip(
                result.boxes.xyxy.cpu().numpy(),
                result.boxes.conf.cpu().numpy(),
            ):
                dets.append({"image_id": image_id, "category_id": 1,
                             "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                             "score": float(score)})

    if not dets:
        conf_results.append({"conf": conf, "map": 0.0, "map50": 0.0})
        continue

    ev = COCOeval(coco_gt, coco_gt.loadRes(dets), "bbox")
    ev.evaluate(); ev.accumulate(); ev.summarize()
    conf_results.append({"conf": conf, "map": ev.stats[0], "map50": ev.stats[1]})
    print(f"conf={conf:.2f}  mAP@0.50:0.95={ev.stats[0]:.4f}  mAP@0.50={ev.stats[1]:.4f}  n_dets={len(dets)}")

BEST_CONF = max(conf_results, key=lambda r: r["map"])["conf"]
print(f"\n>>> Best conf: {BEST_CONF:.2f}")

Loading and preparing results...
DONE (t=0.25s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.72s).
Accumulating evaluation results...
DONE (t=0.10s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.450
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.735
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.492
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.423
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.494
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.317
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.221
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.583
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.647
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=10

In [12]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot([r["conf"] for r in conf_results], [r["map"]   for r in conf_results],
        marker="o", label="mAP@0.50:0.95", color="steelblue")
ax.plot([r["conf"] for r in conf_results], [r["map50"] for r in conf_results],
        marker="s", label="mAP@0.50", color="darkorange", linestyle="--")
ax.axvline(BEST_CONF, color="red", linestyle=":", label=f"best conf={BEST_CONF:.2f}")
ax.axhline(0.4484, color="grey", linestyle="--", alpha=0.5, label="Exp C baseline (0.4484)")
ax.set_xlabel("Confidence threshold")
ax.set_ylabel("mAP")
ax.set_title("Confidence sweep — Exp D RT-DETR-X@640")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

<Figure size 900x400 with 1 Axes>

## Submission — `submission_phase4.json`

Building the test submission with the best confidence threshold from the sweep.

In [13]:
test_paths = sorted(IMG_TEST.glob("*.png"))
print(f"Inferring on {len(test_paths)} test images")
print(f"  Model : Exp D RT-DETR-X  |  conf: {BEST_CONF:.2f}  |  iou: {IOU_NMS}")

detections_d = []
BATCH = 8

for i in range(0, len(test_paths), BATCH):
    preds = model_sweep.predict(
        source  = [str(p) for p in test_paths[i:i+BATCH]],
        imgsz   = 640,
        conf    = BEST_CONF,
        iou     = IOU_NMS,
        device  = 0,
        verbose = False,
    )
    for result, img_path in zip(preds, test_paths[i:i+BATCH]):
        image_id = int(img_path.stem)
        for (x1, y1, x2, y2), score in zip(
            result.boxes.xyxy.cpu().numpy(),
            result.boxes.conf.cpu().numpy(),
        ):
            detections_d.append({"image_id": image_id, "category_id": 1,
                                  "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                                  "score": float(score)})

print(f"Total detections: {len(detections_d)}")
sub_path = Path("submission_phase4.json")
sub_path.write_text(json.dumps(detections_d))
print(f"Saved → {sub_path}")

Inferring on 786 test images
  Model : Exp D RT-DETR-X  |  conf: 0.00  |  iou: 0.5
Total detections: 235800
Saved → submission_phase4.json


In [9]:
# Quick visual overlay on a few test images
sample_ids = [10, 90, 356]
det_by_id  = defaultdict(list)
for d in detections_d:
    det_by_id[d["image_id"]].append(d)

for sid in sample_ids:
    img_path = IMG_TEST / f"{sid}.png"
    if not img_path.exists():
        continue
    img = Image.open(img_path)
    fig, ax = plt.subplots(1, figsize=(11, 4))
    ax.imshow(img)
    for d in det_by_id[sid]:
        x, y, w, h = d["bbox"]
        ax.add_patch(mpatches.Rectangle((x, y), w, h, linewidth=1.5, edgecolor="lime", facecolor="none"))
        ax.text(x, y - 2, f"{d['score']:.2f}", color="lime", fontsize=7, va="bottom")
    ax.set_title(f"Test image {sid} — {len(det_by_id[sid])} detections (RT-DETR-X, conf={BEST_CONF:.2f})")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

<Figure size 1100x400 with 1 Axes>

<Figure size 1100x400 with 1 Axes>

<Figure size 1100x400 with 1 Axes>

In [10]:
del model_sweep
gc.collect()
torch.cuda.empty_cache()
print("Done — memory freed.")

Done — memory freed.


---
## Wrap-up

**Exp D RT-DETR-X reached 0.4503 val mAP / 0.4682 test mAP.** New project best. The val→test gap is +0.0179 in favour of test, which is unusually positive (test was actually slightly easier than val).

A few notes:
- The jump from L (0.4484) to X (0.4503) is small on val (+0.0019) but the test gap suggests X is genuinely the better model — it's likely that the few cases X handles better are over-represented in test
- 0.4682 puts the submission in the top 9 globally, which is more than I expected from a "just scale up the model" experiment
- The model hadn't fully plateaued at epoch 90 — `patience=70` never fired, training stopped at 100. There's probably a sliver more to extract by going longer or using a finer LR finder, which is what Phase 8 will explore

**Looking ahead:**
- Phase 5 — try adding a P2 stride-4 head to X (more capacity for the tiniest stripes — 49% of boxes are <32²)
- Phase 6+ — fine-tuning experiments, more ensembles, TTA variants

If P2 helps it'll be the cleanest path to 0.50. If not, the path forward gets harder.